# Product Survivorship Engine

**Purpose:** Consolidate the source records in each approved product group into one product record by applying the assignment's survivorship rules.

**Lakehouse inputs**
- `silver.products`
- `semantic.finalmatchingdecisions`

**Intermediate output**
- `semantic.productgroupmembers` (created by the Entity Resolution & Product Grouping notebook)

**In-memory output**
- `ConsolidatedProductsDf`

This notebook implements survivorship only. Field-level lineage will be generated in the next notebook using the selected source record metadata retained here.

## Survivorship rules

| Attribute | Rule |
|---|---|
| ProductName | Prefer the longer valid value |
| Description | Prefer the longer valid value |
| Price | Choose the lower valid price |
| Brand, Category, Color, Size, Material, InStock | Prefer the most recently updated valid value |

**Deterministic tie-breaking:** if the primary rule ties, prefer the latest valid `UpdatedAt`; if that also ties or is missing, sort by `SourceSystem` and `SourceProductId` ascending.

Null and empty text values are not considered valid winners when another group member has a valid value. If every member is null for an attribute, the selected value remains null and its source metadata remains null.

In [3]:
# Cell 1 – Imports and load input datasets

from pyspark.sql import functions as F
from pyspark.sql.window import Window

SilverDf = spark.table("silver.products")

# The grouping notebook must write this intermediate mapping before this notebook runs.
GroupMembersDf = spark.table("semantic.productgroupmembers")

print("Silver products:", SilverDf.count())
print("Product group members:", GroupMembersDf.count())

StatementMeta(, 2d8e77c8-9ede-4288-8562-f1c4e800ade6, 5, Finished, Available, Finished, False)

Silver products: 164
Product group members: 164


In [7]:
# Cell 2 – Prepare source records and join them to their product groups

RequiredSilverColumns = {
    "SourceSystem", "SourceProductID", "ProductName", "Description",
    "Price", "Brand", "Category", "Color", "Size", "Material",
    "InStock", "UpdatedAt"
}
MissingColumns = RequiredSilverColumns - set(SilverDf.columns)

if MissingColumns:
    raise ValueError(f"Missing expected Silver columns: {sorted(MissingColumns)}")

RequiredGroupColumns = {
    "GoldProductGroupId", "SourceSystem", "SourceProductId"
}
MissingGroupColumns = RequiredGroupColumns - set(GroupMembersDf.columns)

if MissingGroupColumns:
    raise ValueError(f"Missing expected group mapping columns: {sorted(MissingGroupColumns)}")

GroupedProductsDf = (
    GroupMembersDf
    .select("GoldProductGroupId", "SourceSystem", "SourceProductId")
    .join(
        SilverDf,
        on=["SourceSystem", "SourceProductId"],
        how="inner"
    )
    .withColumn(
        "_UpdatedAtTs",
        F.coalesce(
            F.to_timestamp("UpdatedAt"),
            F.to_timestamp("UpdatedAt", "yyyy-MM-dd'T'HH:mm:ss"),
            F.to_timestamp("UpdatedAt", "yyyy-MM-dd HH:mm:ss")
        )
    )
)

print("Grouped source records:", GroupedProductsDf.count())
display(GroupedProductsDf.limit(10))

StatementMeta(, 2d8e77c8-9ede-4288-8562-f1c4e800ade6, 9, Finished, Available, Finished, False)

Grouped source records: 164


SynapseWidget(Synapse.DataFrame, db82c035-2a2d-4227-ba87-34ca2c244764)

## Winner-selection implementation

The helper below uses Spark window functions (not Python UDFs). It returns the selected value together with the source system, source product ID, source update timestamp, and the rule applied. Retaining this provenance now makes the next field-level lineage step straightforward.

In [8]:
# Cell 3 – Reusable deterministic survivorship function

def SelectWinningAttribute(
    InputDf,
    AttributeName,
    RuleType
):
    """
    RuleType:
      LONGEST  -> longer non-empty string wins
      LOWEST   -> lowest valid numeric value wins
      LATEST   -> most recently updated valid value wins
    """

    ValueColumn = F.col(AttributeName)

    if RuleType == "LONGEST":
        IsValid = ValueColumn.isNotNull() & (F.length(F.trim(ValueColumn.cast("string"))) > 0)
        PrimarySort = F.length(F.trim(ValueColumn.cast("string"))).desc()

    elif RuleType == "LOWEST":
        IsValid = ValueColumn.isNotNull()
        PrimarySort = ValueColumn.asc()

    elif RuleType == "LATEST":
        if AttributeName in ["Brand", "Category", "Color", "Size", "Material"]:
            IsValid = ValueColumn.isNotNull() & (F.length(F.trim(ValueColumn.cast("string"))) > 0)
        else:
            IsValid = ValueColumn.isNotNull()
        PrimarySort = F.col("_UpdatedAtTs").desc_nulls_last()

    else:
        raise ValueError(f"Unsupported survivorship rule: {RuleType}")

    CandidatesDf = InputDf.filter(IsValid)

    OrderExpressions = [PrimarySort]

    if RuleType != "LATEST":
        OrderExpressions.append(F.col("_UpdatedAtTs").desc_nulls_last())

    OrderExpressions.extend([
        F.col("SourceSystem").asc(),
        F.col("SourceProductId").asc()
    ])

    AttributeWindow = (
        Window
        .partitionBy("GoldProductGroupId")
        .orderBy(*OrderExpressions)
    )

    WinnersDf = (
        CandidatesDf
        .withColumn("_WinnerRank", F.row_number().over(AttributeWindow))
        .filter(F.col("_WinnerRank") == 1)
        .select(
            "GoldProductGroupId",
            F.col(AttributeName).alias(AttributeName),
            F.col("SourceSystem").alias(f"{AttributeName}SourceSystem"),
            F.col("SourceProductId").alias(f"{AttributeName}SourceProductID"),
            F.col("UpdatedAt").alias(f"{AttributeName}SourceUpdatedAt"),
            F.lit(RuleType).alias(f"{AttributeName}SurvivorshipRule")
        )
    )

    return WinnersDf

StatementMeta(, 2d8e77c8-9ede-4288-8562-f1c4e800ade6, 10, Finished, Available, Finished, False)

In [9]:
# Cell 4 – Apply survivorship rules to all required attributes

AttributeRules = {
    "ProductName": "LONGEST",
    "Description": "LONGEST",
    "Price": "LOWEST",
    "Brand": "LATEST",
    "Category": "LATEST",
    "Color": "LATEST",
    "Size": "LATEST",
    "Material": "LATEST",
    "InStock": "LATEST"
}

AttributeWinnerDfs = {}

for AttributeName, RuleType in AttributeRules.items():
    AttributeWinnerDfs[AttributeName] = SelectWinningAttribute(
        GroupedProductsDf,
        AttributeName,
        RuleType
    )

    print(
        f"{AttributeName}: "
        f"{AttributeWinnerDfs[AttributeName].count()} winning records"
    )

StatementMeta(, 2d8e77c8-9ede-4288-8562-f1c4e800ade6, 11, Finished, Available, Finished, False)

ProductName: 142 winning records
Description: 142 winning records
Price: 142 winning records
Brand: 142 winning records
Category: 142 winning records
Color: 142 winning records
Size: 142 winning records
Material: 142 winning records
InStock: 142 winning records


In [10]:
# Cell 5 – Assemble one consolidated record per product group

GroupIdsDf = GroupMembersDf.select("GoldProductGroupId").distinct()

ConsolidatedProductsDf = GroupIdsDf

for AttributeName in AttributeRules.keys():
    ConsolidatedProductsDf = ConsolidatedProductsDf.join(
        AttributeWinnerDfs[AttributeName],
        on="GoldProductGroupId",
        how="left"
    )

print("Consolidated product groups:", ConsolidatedProductsDf.count())
display(ConsolidatedProductsDf)

StatementMeta(, 2d8e77c8-9ede-4288-8562-f1c4e800ade6, 12, Finished, Available, Finished, False)

Consolidated product groups: 142


SynapseWidget(Synapse.DataFrame, 7dca4f87-f7ae-48d4-a037-c64fc021253e)

In [11]:
# Cell 6 – Add group membership summary

GroupSummaryDf = (
    GroupMembersDf
    .groupBy("GoldProductGroupId")
    .agg(
        F.count("*").alias("SourceRecordCount"),
        F.collect_set("SourceSystem").alias("ContributingSourceSystems")
    )
)

ConsolidatedProductsDf = (
    ConsolidatedProductsDf
    .join(GroupSummaryDf, on="GoldProductGroupId", how="left")
)

display(ConsolidatedProductsDf.limit(20))

StatementMeta(, 2d8e77c8-9ede-4288-8562-f1c4e800ade6, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4d2dcfd7-1922-4dda-ab07-97d140df1b25)

## Validation built into the implementation

The following checks confirm one consolidated row per group and show whether any attribute has no valid source value. These are implementation checks, not a substitute for reviewing the actual output.

In [12]:
# Cell 7 – Survivorship output checks

TotalGroups = GroupMembersDf.select("GoldProductGroupId").distinct().count()
ConsolidatedCount = ConsolidatedProductsDf.count()

DuplicateGoldGroupsDf = (
    ConsolidatedProductsDf
    .groupBy("GoldProductGroupId")
    .count()
    .filter(F.col("count") > 1)
)

print("Distinct input groups:", TotalGroups)
print("Consolidated output rows:", ConsolidatedCount)
print("Duplicate consolidated group IDs:", DuplicateGoldGroupsDf.count())

if ConsolidatedCount != TotalGroups:
    raise ValueError(
        f"Expected one output row per group ({TotalGroups}), "
        f"but produced {ConsolidatedCount}."
    )

if DuplicateGoldGroupsDf.limit(1).count() > 0:
    display(DuplicateGoldGroupsDf)
    raise ValueError("More than one consolidated row exists for a product group.")

NullAttributeCounts = ConsolidatedProductsDf.agg(*[
    F.sum(F.when(F.col(AttributeName).isNull(), 1).otherwise(0)).alias(AttributeName)
    for AttributeName in AttributeRules.keys()
])

display(NullAttributeCounts)

print("SURVIVORSHIP OUTPUT STRUCTURE VALIDATION PASSED.")

StatementMeta(, 2d8e77c8-9ede-4288-8562-f1c4e800ade6, 14, Finished, Available, Finished, False)

Distinct input groups: 142
Consolidated output rows: 142
Duplicate consolidated group IDs: 0


SynapseWidget(Synapse.DataFrame, d435dd74-4305-4c79-a0f1-a0b01628f3ea)

SURVIVORSHIP OUTPUT STRUCTURE VALIDATION PASSED.


In [13]:
ConsolidatedProductsDf.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold.consolidatedproducts")

StatementMeta(, 2d8e77c8-9ede-4288-8562-f1c4e800ade6, 15, Finished, Available, Finished, False)